# 🌦️ Pipeline Climático con NumPy
## Ejercicio de mundo real — Rol: Data Engineer / Analista de Datos

---

### 🏢 Contexto del negocio

Trabajas como **Data Engineer** en **LogiMex**, una empresa de logística y transporte de carga que opera rutas terrestres entre las principales ciudades de México.

El equipo de operaciones necesita un **reporte diario de condiciones climáticas** en cada ciudad de la red, porque:

- Temperaturas extremas afectan ciertos tipos de carga (farmacéutica, alimentos)
- Velocidades de viento altas complican la operación de grúas en centros de distribución
- La precipitación impacta los tiempos de entrega en zonas urbanas

Tu tarea: construir un **pipeline ETL** (Extract → Transform → Load/Report) usando datos reales de una API meteorológica pública y NumPy para las transformaciones.

---

### 🗺️ Ciudades de la red LogiMex

| Ciudad | Latitud | Longitud |
|--------|---------|----------|
| Ciudad de México | 19.43 | -99.13 |
| Guadalajara | 20.66 | -103.35 |
| Monterrey | 25.67 | -100.31 |
| León | 21.12 | -101.68 |
| Puebla | 19.04 | -98.20 |
| Mérida | 20.97 | -89.62 |
| Tijuana | 32.53 | -117.04 |
| Hermosillo | 29.07 | -110.95 |

**API que usaremos:** [Open-Meteo](https://open-meteo.com/) — gratuita, sin API key, datos reales en tiempo real.

---

```
EXTRACT          TRANSFORM                   REPORT
  │                  │                          │
Open-Meteo API → numpy arrays → operaciones → alertas + ranking
  │               por ciudad    vectorizadas     operativo
  └─ JSON raw   └─ matriz 8×24               └─ tabla resumen
```

In [ ]:
# ============================================================
# SETUP — librerías necesarias
# ============================================================
import numpy as np
import requests
import json
from datetime import datetime, timezone

print(f"NumPy {np.__version__} listo ✓")
print(f"Fecha/hora UTC: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')}")

---
## FASE 1 — EXTRACT 📡
### Consumir la API de Open-Meteo

Open-Meteo expone un endpoint REST que regresa datos horarios de las últimas 24 horas para cualquier coordenada geográfica. No necesitas registrarte ni obtener un token.

**Endpoint base:**
```
https://api.open-meteo.com/v1/forecast
```

**Variables que vamos a pedir:**
- `temperature_2m` — temperatura en °C a 2 metros del suelo
- `windspeed_10m` — velocidad del viento en km/h a 10 metros
- `precipitation` — precipitación en mm/h
- `relativehumidity_2m` — humedad relativa en %

In [ ]:
# ============================================================
# Definir las ciudades de la red LogiMex
# ============================================================
CIUDADES = [
    "CDMX", "Guadalajara", "Monterrey", "León",
    "Puebla", "Mérida", "Tijuana", "Hermosillo"
]

COORDENADAS = {
    #           lat      lon
    "CDMX":        (19.43,  -99.13),
    "Guadalajara": (20.66, -103.35),
    "Monterrey":   (25.67, -100.31),
    "León":        (21.12, -101.68),
    "Puebla":      (19.04,  -98.20),
    "Mérida":      (20.97,  -89.62),
    "Tijuana":     (32.53, -117.04),
    "Hermosillo":  (29.07, -110.95),
}

print(f"Red LogiMex: {len(CIUDADES)} ciudades configuradas ✓")

In [ ]:
# ============================================================
# Función para llamar la API por ciudad
# ============================================================
def fetch_weather(ciudad: str) -> dict:
    """
    Llama la API de Open-Meteo y regresa el JSON con datos
    horarios de las últimas 24 horas para una ciudad dada.
    """
    lat, lon = COORDENADAS[ciudad]

    params = {
        "latitude":  lat,
        "longitude": lon,
        "hourly": "temperature_2m,windspeed_10m,precipitation,relativehumidity_2m",
        "forecast_days": 1,          # solo el día de hoy
        "timezone": "America/Mexico_City"
    }

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params=params,
        timeout=10
    )
    response.raise_for_status()      # lanza excepción si la API falla
    return response.json()

# Probemos con una ciudad antes de hacer el loop completo
test = fetch_weather("León")
print("Estructura del JSON recibido:")
print(json.dumps(list(test.keys()), indent=2))
print("\nVariables horarias disponibles:")
print(list(test["hourly"].keys()))
print(f"\nNúmero de registros por variable: {len(test['hourly']['temperature_2m'])}")

In [ ]:
# ============================================================
# Extraer datos para TODAS las ciudades
# ============================================================
print("Consultando API para cada ciudad...")
raw_data = {}

for ciudad in CIUDADES:
    try:
        raw_data[ciudad] = fetch_weather(ciudad)
        print(f"  ✓ {ciudad}")
    except Exception as e:
        print(f"  ✗ {ciudad} — ERROR: {e}")

print(f"\nExtracción completada: {len(raw_data)}/{len(CIUDADES)} ciudades")

---
## FASE 2 — TRANSFORM 🔧
### Convertir JSON crudo en matrices NumPy

El JSON que recibimos de la API no es directamente operable. El primer paso del transform es estructurar los datos en matrices NumPy con las que podamos trabajar eficientemente.

**Estructura objetivo:**

```
temperatura_matrix  → shape (8 ciudades, 24 horas)
viento_matrix       → shape (8 ciudades, 24 horas)
precipitacion_matrix→ shape (8 ciudades, 24 horas)
humedad_matrix      → shape (8 ciudades, 24 horas)
```

Cada **fila** = una ciudad. Cada **columna** = una hora del día (0h a 23h).

In [ ]:
# ============================================================
# T1 — Estructurar los datos en matrices NumPy
# ============================================================
N_CIUDADES = len(CIUDADES)
N_HORAS    = 24

# Inicializamos con NaN para detectar si algo falló en la extracción
temperatura_mx   = np.full((N_CIUDADES, N_HORAS), np.nan)
viento_mx        = np.full((N_CIUDADES, N_HORAS), np.nan)
precipitacion_mx = np.full((N_CIUDADES, N_HORAS), np.nan)
humedad_mx       = np.full((N_CIUDADES, N_HORAS), np.nan)

for i, ciudad in enumerate(CIUDADES):
    if ciudad not in raw_data:
        continue
    h = raw_data[ciudad]["hourly"]
    temperatura_mx[i]   = h["temperature_2m"][:N_HORAS]
    viento_mx[i]        = h["windspeed_10m"][:N_HORAS]
    precipitacion_mx[i] = h["precipitation"][:N_HORAS]
    humedad_mx[i]       = h["relativehumidity_2m"][:N_HORAS]

print("Matrices construidas:")
print(f"  temperatura   : {temperatura_mx.shape}  dtype={temperatura_mx.dtype}")
print(f"  viento        : {viento_mx.shape}")
print(f"  precipitación : {precipitacion_mx.shape}")
print(f"  humedad       : {humedad_mx.shape}")

print(f"\nDatos faltantes (NaN): {np.isnan(temperatura_mx).sum()}")

In [ ]:
# Vista rápida de la matriz de temperatura
# Filas = ciudades, columnas = horas 0h-23h
print("Temperatura (°C) por ciudad y hora:")
print(f"{'':>15}", end="")
for h in range(0, 24, 3):          # solo cada 3 horas para que entre en pantalla
    print(f"  {h:02d}h", end="")
print()

for i, ciudad in enumerate(CIUDADES):
    print(f"{ciudad:>15}", end="")
    for h in range(0, 24, 3):
        print(f"  {temperatura_mx[i, h]:>4.1f}", end="")
    print()

### T2 — Conversiones de unidades

El equipo de operaciones en campo trabaja con:
- Temperatura en **°F** (para rutas hacia frontera norte / clientes en EE.UU.)
- Viento en **m/s** (estándar en reportes de seguridad)

NumPy hace estas conversiones de forma vectorizada sobre toda la matriz al mismo tiempo.

In [ ]:
# ============================================================
# T2 — Conversiones de unidades (vectorizadas)
# ============================================================

# °C → °F : F = C * 9/5 + 32
temperatura_F = temperatura_mx * (9/5) + 32

# km/h → m/s : ms = kmh / 3.6
viento_ms = viento_mx / 3.6

# Verificamos con Mérida, hora 14 (la más caliente del día)
idx_merida = CIUDADES.index("Mérida")
print("Verificación conversión (Mérida, 14h):")
print(f"  {temperatura_mx[idx_merida, 14]:.1f} °C  →  {temperatura_F[idx_merida, 14]:.1f} °F")
print(f"  {viento_mx[idx_merida, 14]:.1f} km/h  →  {viento_ms[idx_merida, 14]:.2f} m/s")

# Nota: no se escribió ningún loop — NumPy aplicó la operación
# a los 8*24 = 192 valores simultáneamente

### T3 — Estadísticas por ciudad (agregaciones con axis)

Para el reporte operativo necesitamos los resúmenes del **día** (agregar las 24 horas) para cada ciudad.

In [ ]:
# ============================================================
# T3 — Estadísticas diarias por ciudad
# axis=1 → colapsa las 24 horas → resultado shape (8,)
# ============================================================
temp_media    = np.mean(temperatura_mx, axis=1)     # promedio del día
temp_max      = np.max(temperatura_mx,  axis=1)     # máxima del día
temp_min      = np.min(temperatura_mx,  axis=1)     # mínima del día
temp_rango    = temp_max - temp_min                 # amplitud térmica

viento_max    = np.max(viento_mx,       axis=1)
viento_media  = np.mean(viento_mx,      axis=1)

precip_total  = np.sum(precipitacion_mx, axis=1)   # mm acumulados
humedad_media = np.mean(humedad_mx,      axis=1)

print(f"{'Ciudad':>13}  {'T.media':>7}  {'T.max':>5}  {'T.min':>5}  {'Rango':>5}  {'V.max':>5}  {'Precip':>6}")
print("-" * 70)
for i, ciudad in enumerate(CIUDADES):
    print(f"{ciudad:>13}  {temp_media[i]:>6.1f}°C  "
          f"{temp_max[i]:>4.1f}°  "
          f"{temp_min[i]:>4.1f}°  "
          f"{temp_rango[i]:>4.1f}°  "
          f"{viento_max[i]:>4.1f}k  "
          f"{precip_total[i]:>5.1f}mm")

### T4 — Índice de Riesgo Operativo

El equipo de operaciones definió un **Índice de Riesgo** compuesto que combina temperatura, viento y lluvia en un solo número entre 0 y 100.

La fórmula es:

$$\text{IRO} = 0.4 \cdot T_{norm} + 0.4 \cdot V_{norm} + 0.2 \cdot P_{norm}$$

Donde cada variable se **normaliza** al rango [0, 1] respecto a los valores máximos observados en la red. Aquí usaremos broadcasting para normalizar todas las ciudades a la vez.

In [ ]:
# ============================================================
# T4 — Índice de Riesgo Operativo (IRO)
# ============================================================

# Normalización min-max respecto al máximo de la red
# Broadcasting: dividir cada escalar entre su máximo global
def normalizar(arr):
    """Normaliza un array 1D al rango [0, 1]"""
    mn, mx = arr.min(), arr.max()
    if mx == mn:
        return np.zeros_like(arr, dtype=float)
    return (arr - mn) / (mx - mn)

T_norm = normalizar(temp_max)       # temperatura máxima normalizada
V_norm = normalizar(viento_max)     # viento máximo normalizado
P_norm = normalizar(precip_total)   # precipitación total normalizada

# Índice compuesto — operación vectorizada sobre los 8 valores
IRO = 0.4 * T_norm + 0.4 * V_norm + 0.2 * P_norm
IRO_pct = IRO * 100   # llevamos a escala 0-100

print("Índice de Riesgo Operativo por ciudad:")
print()
# Ordenar de mayor a menor riesgo con argsort
orden_riesgo = np.argsort(IRO_pct)[::-1]   # descendente

for rank, i in enumerate(orden_riesgo, 1):
    barra = "█" * int(IRO_pct[i] / 5)      # barra visual proporcional
    nivel = "🔴 ALTO " if IRO_pct[i] > 60 else ("🟡 MEDIO" if IRO_pct[i] > 30 else "🟢 BAJO ")
    print(f"  #{rank} {CIUDADES[i]:>13} [{nivel}] {IRO_pct[i]:>5.1f}/100  {barra}")

### T5 — Detección de Alertas con Boolean Indexing

Definimos umbrales de alerta por variable. Usamos boolean indexing para detectar qué ciudades los superan.

In [ ]:
# ============================================================
# T5 — Sistema de alertas operativas
# ============================================================
UMBRAL_TEMP_ALTA   = 35.0   # °C — riesgo para carga farmacéutica
UMBRAL_TEMP_BAJA   = 5.0    # °C — riesgo para carga perecedera
UMBRAL_VIENTO      = 50.0   # km/h — suspender operaciones de grúa
UMBRAL_PRECIP      = 5.0    # mm  — alerta por lluvia acumulada

# Boolean masks — operaciones vectorizadas
alerta_calor   = temp_max    > UMBRAL_TEMP_ALTA
alerta_frio    = temp_min    < UMBRAL_TEMP_BAJA
alerta_viento  = viento_max  > UMBRAL_VIENTO
alerta_lluvia  = precip_total > UMBRAL_PRECIP

# Ciudades con al menos una alerta activa (OR entre masks)
cualquier_alerta = alerta_calor | alerta_frio | alerta_viento | alerta_lluvia

print("=" * 55)
print("       REPORTE DE ALERTAS LOGIMEX")
print("=" * 55)

ciudades_np = np.array(CIUDADES)

if cualquier_alerta.any():
    for i in np.where(cualquier_alerta)[0]:   # fancy indexing
        alertas_activas = []
        if alerta_calor[i]:
            alertas_activas.append(f"🌡️  Calor extremo ({temp_max[i]:.1f}°C > {UMBRAL_TEMP_ALTA}°C)")
        if alerta_frio[i]:
            alertas_activas.append(f"❄️  Frío extremo ({temp_min[i]:.1f}°C < {UMBRAL_TEMP_BAJA}°C)")
        if alerta_viento[i]:
            alertas_activas.append(f"💨  Viento alto ({viento_max[i]:.1f} km/h > {UMBRAL_VIENTO} km/h)")
        if alerta_lluvia[i]:
            alertas_activas.append(f"🌧️  Lluvia acumulada ({precip_total[i]:.1f} mm > {UMBRAL_PRECIP} mm)")

        print(f"\n📍 {CIUDADES[i]}")
        for a in alertas_activas:
            print(f"   → {a}")
else:
    print("\n✅ Sin alertas activas en la red")

print(f"\nCiudades sin alertas: {ciudades_np[~cualquier_alerta].tolist()}")

### T6 — Análisis de Horas Críticas

¿A qué horas del día se presentan las condiciones más adversas? Esto le sirve a operaciones para **programar horarios de carga/descarga** fuera de esas ventanas.

In [ ]:
# ============================================================
# T6 — Análisis de horas críticas en la red completa
# ============================================================

# Temperatura promedio de la RED por hora (promediar sobre ciudades)
# axis=0 → colapsa las 8 ciudades → resultado shape (24,)
temp_red_por_hora    = np.mean(temperatura_mx, axis=0)
viento_red_por_hora  = np.mean(viento_mx,      axis=0)
precip_red_por_hora  = np.mean(precipitacion_mx, axis=0)

# Hora de temperatura máxima y mínima promedio en la red
hora_mas_caliente = np.argmax(temp_red_por_hora)
hora_mas_fria     = np.argmin(temp_red_por_hora)
hora_mas_ventosa  = np.argmax(viento_red_por_hora)

print("Promedios horarios en la red (°C / km/h / mm):")
print()
print(f"{'Hora':>5}  {'Temp':>6}  {'Viento':>7}  {'Precip':>7}")
print("-" * 35)
for h in range(24):
    marcador = ""
    if h == hora_mas_caliente: marcador = " ← 🌡️  MAX"
    if h == hora_mas_fria:     marcador = " ← ❄️  MIN"
    if h == hora_mas_ventosa and h not in [hora_mas_caliente, hora_mas_fria]:
        marcador = " ← 💨 VIENTO"
    print(f"  {h:02d}h  {temp_red_por_hora[h]:>5.1f}°C  "
          f"{viento_red_por_hora[h]:>5.1f}km/h  "
          f"{precip_red_por_hora[h]:>5.2f}mm"
          f"{marcador}")

In [ ]:
# Ventana operativa recomendada
# Horas donde la temperatura de la RED está bajo el percentil 40
# (condiciones más frescas para carga perecedera)

umbral_fresco = np.percentile(temp_red_por_hora, 40)
horas_optimas = np.where(temp_red_por_hora <= umbral_fresco)[0]

print(f"Umbral de temperatura 'fresca' (p40): {umbral_fresco:.1f}°C")
print(f"Horas óptimas para carga perecedera: {[f'{h:02d}h' for h in horas_optimas]}")

---
## FASE 3 — LOAD / REPORT 📊
### Generar el reporte ejecutivo

El último paso del pipeline es presentar los resultados de forma que el equipo de operaciones pueda consumirlos de un vistazo.

In [ ]:
# ============================================================
# REPORTE EJECUTIVO FINAL
# ============================================================
ahora = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

print("╔" + "═" * 57 + "╗")
print("║" + " REPORTE CLIMÁTICO OPERATIVO — LOGIMEX ".center(57) + "║")
print("║" + ahora.center(57) + "║")
print("╠" + "═" * 57 + "╣")

# Ciudad más caliente y más fría
idx_max_temp = np.argmax(temp_max)
idx_min_temp = np.argmin(temp_min)
idx_mas_viento = np.argmax(viento_max)
idx_mas_lluvia = np.argmax(precip_total)
idx_mayor_riesgo = np.argmax(IRO_pct)
idx_menor_riesgo = np.argmin(IRO_pct)

print(f"║ 🌡️  Más caliente   : {CIUDADES[idx_max_temp]:>13} ({temp_max[idx_max_temp]:.1f}°C)  ║")
print(f"║ ❄️  Más fría       : {CIUDADES[idx_min_temp]:>13} ({temp_min[idx_min_temp]:.1f}°C)   ║")
print(f"║ 💨 Mayor viento   : {CIUDADES[idx_mas_viento]:>13} ({viento_max[idx_mas_viento]:.1f} km/h) ║")
print(f"║ 🌧️  Mayor lluvia   : {CIUDADES[idx_mas_lluvia]:>13} ({precip_total[idx_mas_lluvia]:.1f} mm)    ║")
print("╠" + "═" * 57 + "╣")
print(f"║ 🔴 Mayor riesgo   : {CIUDADES[idx_mayor_riesgo]:>13} (IRO={IRO_pct[idx_mayor_riesgo]:.1f}/100)  ║")
print(f"║ 🟢 Menor riesgo   : {CIUDADES[idx_menor_riesgo]:>13} (IRO={IRO_pct[idx_menor_riesgo]:.1f}/100)   ║")
print("╠" + "═" * 57 + "╣")
print(f"║ ⏰ Ventana óptima : {str([f'{h:02d}h' for h in horas_optimas]):<38}║")
print(f"║ 🚨 Ciudades con alertas: {int(cualquier_alerta.sum()):<4} de {N_CIUDADES:<22}║")
print("╚" + "═" * 57 + "╝")

---
## 🏋️ Ejercicios de práctica

Ahora que tienes el pipeline funcionando, extiéndelo con estos retos:

---

### Ejercicio 1 — Amplitud térmica extrema

Agrega al reporte la ciudad con **mayor amplitud térmica** (diferencia entre temperatura máxima y mínima del día). Esto es relevante para rutas donde los camiones pasan de zonas cálidas a zonas frías en el mismo trayecto.

> **Pista:** ya tienes `temp_rango` calculado. Usa `np.argmax()`.

In [ ]:
# Ejercicio 1 — Tu código aquí


### Ejercicio 2 — Índice de Confort del Conductor

El sindicato de conductores pide que se monitoree el **Índice de Confort** en la cabina, definido como:

$$\text{ICC} = T_{media} + 0.33 \cdot e^{0.1 \cdot H_{media}} - 4$$

Donde $T_{media}$ es la temperatura media del día en °C y $H_{media}$ es la humedad relativa media en %. Un ICC > 30 indica incomodidad severa.

Calcula el ICC para cada ciudad y lista cuáles superan el umbral.

> **Pista:** Usa `np.exp()` para la exponencial, y boolean indexing para filtrar.

In [ ]:
# Ejercicio 2 — Tu código aquí


### Ejercicio 3 — Correlación temperatura / humedad

¿Las ciudades más calientes son también las más húmedas? Calcula la correlación entre `temp_media` y `humedad_media` usando `np.corrcoef()`.

Interpreta el resultado: ¿correlación positiva, negativa, o sin relación?

> **Pista:** `np.corrcoef(a, b)` regresa una matriz 2×2; el coeficiente de correlación está en la posición `[0, 1]`.

In [ ]:
# Ejercicio 3 — Tu código aquí


### Ejercicio 4 — Ruta más segura del día

LogiMex tiene dos rutas principales:
- **Ruta Norte:** CDMX → León → Monterrey → Tijuana
- **Ruta Sur:** CDMX → Puebla → Mérida

Calcula el IRO promedio de cada ruta y determina cuál es operativamente más segura hoy.

> **Pista:** Usa fancy indexing para seleccionar las ciudades de cada ruta del array `IRO_pct`.

In [ ]:
# Ejercicio 4 — Tu código aquí
RUTA_NORTE = ["CDMX", "León", "Monterrey", "Tijuana"]
RUTA_SUR   = ["CDMX", "Puebla", "Mérida"]

# Obtener los índices de cada ciudad en la lista CIUDADES
# idx_norte = ...
# idx_sur   = ...


---
## 📝 Resumen del pipeline

```
EXTRACT                  TRANSFORM                         REPORT
──────────────────────   ──────────────────────────────    ───────────────────
Open-Meteo API           T1: JSON → matrices NumPy         Reporte ejecutivo
8 ciudades               T2: Conversiones de unidades      Ranking de riesgo
4 variables              T3: Agregaciones (axis=1/0)       Alertas activas
24 horas cada una        T4: Índice de Riesgo (IRO)        Ventana horaria
                         T5: Alertas (boolean indexing)
                         T6: Análisis horario
```

### Conceptos de NumPy que practicaste

| Concepto | Dónde se usó |
|----------|--------------|
| `np.full(shape, np.nan)` | Inicialización segura de matrices |
| Operaciones vectorizadas | Conversión °C→°F y km/h→m/s sobre toda la matriz |
| `np.mean / max / min / sum` con `axis` | Estadísticas por ciudad (axis=1) y por hora (axis=0) |
| Broadcasting | Normalización de arrays para el IRO |
| `np.argsort()` | Ranking de ciudades por riesgo |
| Boolean indexing | Detección de alertas por umbrales |
| `np.where()` | Detección de horas óptimas |
| `np.argmax() / argmin()` | Ciudad extrema en cada variable |
| Fancy indexing | Selección de ciudades por índice |
| `np.percentile()` | Umbral dinámico de temperatura fresca |